# Migrate to Apache Iceberg
## Full Table Migration of Delta Lake to Iceberg tables in S3 Tables

### Configure Spark Session
Change `<ACCOUNT_ID>`, `<REGION>`, `<TABLE_BUCKET_NAME>` with your S3 tables bucket details

In [ ]:
%%configure -f
{   
    "conf":{
        "numExecutors": 1, 
         "executorMemory": "14G",
         "executorCores": 4, 
        "spark.dynamicAllocation.enabled": "false",
        "spark.sql.catalog.s3tablesbucket":"org.apache.iceberg.spark.SparkCatalog",
        "spark.sql.catalog.s3tablesbucket.catalog-impl":"org.apache.iceberg.aws.glue.GlueCatalog",
        "spark.sql.catalog.s3tablesbucket.glue.catalog-arn":"arn:aws:glue:<REGION>:<ACCOUNT_ID>:catalog/s3tablescatalog/<TABLE_BUCKET_NAME>",
        "spark.sql.catalog.s3tablesbucket.glue.id":"<ACCOUNT_ID>:s3tablescatalog/<TABLE_BUCKET_NAME>",
        "spark.sql.catalog.s3tablesbucket.warehouse":"arn:aws:s3tables:<REGION>:<ACCOUNT_ID>:bucket/<TABLE_BUCKET_NAME>",
        "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension",
        "spark.jars": "/usr/share/aws/delta/lib/delta-spark.jar,/usr/share/aws/delta/lib/delta-storage.jar",
        "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    }
}  

### Configure variables

In [ ]:
source_table_name = "events_2_delta"
target_table_name="events_2_s3tables_from_delta"
bucket_name="mybucket"
db_name="mydb"
namespace_name="my_namespace"

### Create db

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {db_name}
LOCATION 's3://{bucket_name}/{db_name}'
""")

### Create namespace

In [ ]:
spark.sql(f"""
CREATE NAMESPACE IF NOT EXISTS s3tablesbucket.{namespace_name}
""")

### Create and populate source table

In [ ]:
spark.sql(f"""
CREATE TABLE {db_name}.{source_table_name} (
    event_id STRING,
    user_id STRING,
    event_type STRING,
    event_timestamp TIMESTAMP,
    additional_data STRING,
    year INT,
    month INT,
    day INT
)
USING DELTA
PARTITIONED BY (year,month,day)
LOCATION 's3://{bucket_name}/{db_name}/{source_table_name}/'
""")

In [ ]:
spark.sql(f"""
INSERT INTO {db_name}.{source_table_name}
VALUES 
    ('EVT_001', 'USER_006', 'click', timestamp('2024-04-01 09:15:00'), 'data1', 2024, 4, 1),
    ('EVT_002', 'USER_003', 'purchase', timestamp('2024-04-01 17:30:00'), 'data2', 2024, 4, 1),
    ('EVT_003', 'USER_005', 'view', timestamp('2024-04-02 11:45:00'), 'data3', 2024, 4, 2),
    ('EVT_004', 'USER_001', 'login', timestamp('2024-04-02 14:20:00'), 'data4', 2024, 4, 2),
    ('EVT_005', 'USER_004', 'click', timestamp('2024-04-03 10:10:00'), 'data5', 2024, 4, 3),
    ('EVT_006', 'USER_002', 'purchase', timestamp('2024-04-03 16:30:00'), 'data6', 2024, 4, 3)
""")

Verify Delta Lake table

In [ ]:
spark.sql(f"""
SELECT * FROM {db_name}.{source_table_name} ORDER BY event_id
""").show(truncate=False)

### Full migration with CREATE + INSERT

Let's perform full migration to Iceberg in S3 Tables. We also implement Iceberg hidden partitioning during the migration.

In [ ]:
spark.sql(f"""
CREATE TABLE s3tablesbucket.{namespace_name}.{target_table_name} (
    event_id STRING,
    user_id STRING,
    event_type STRING,
    event_timestamp TIMESTAMP,
    additional_data STRING
)
USING iceberg
PARTITIONED BY (days(event_timestamp))
""")

In [ ]:
spark.sql(f"""
INSERT INTO s3tablesbucket.{namespace_name}.{target_table_name}
SELECT 
    event_id,
    user_id,
    event_type,
    event_timestamp,
    additional_data
FROM {db_name}.{source_table_name}
""")

Verify Iceberg table

In [ ]:
spark.sql(f"""
SELECT * FROM s3tablesbucket.{namespace_name}.{target_table_name} ORDER BY event_id
""").show(truncate=False)